# Mini-FANET reproductible : mobilité, rupture et reroutage

Ce notebook construit une expérience pédagogique complète en **Python pur** : quatre drones échangent avec une station sol, un relais quitte brutalement sa position, la route active casse, puis le plan de contrôle installe un chemin de secours. On mesure ensuite la livraison, le délai, les pertes, les sauts et le temps de convergence.

**Public cible :** étudiant, alternant ou technicien débutant sachant lire un peu de Python et un graphe. **Durée :** 45 minutes. Le livrable attendu est le tableau de bord expliqué, puis une variante de portée ou de convergence dont l'effet est justifié.

L'objectif est de rendre visibles les mécanismes, pas d'imiter toute une pile radio. Le modèle est **synthétique, déterministe et non destiné à une qualification opérationnelle**. Il se rapproche de l'esprit du noyau `network_simulation_labs` : événements explicites, plus court chemin, métriques vérifiables et aucune dépendance réseau.

In [ ]:
from __future__ import annotations

import hashlib
import heapq
from html import escape
import json
import math
import random
import statistics

SEED = 20_260_730
DURATION_S = 60
PACKETS_PER_S = 10
RADIO_RANGE_M = 165.0
CONTROL_DELAY_S = 3
SOURCE, DESTINATION = "S", "G"
NODES = ("G", "S", "R1", "U1", "U2")

print(f"Seed={SEED} | horizon={DURATION_S} s | débit={PACKETS_PER_S} paquets/s | stdlib-only")

## 1. Modèle minimal, hypothèses lisibles

- `G` est la station au sol ; `S` est le drone source.
- `R1` forme normalement le chemin court `S → R1 → G`. Entre 20 s et 40 s, il s'écarte pour représenter une mission ou une mobilité imprévue.
- `U1` et `U2` forment un corridor de secours plus long.
- Deux nœuds sont voisins si leur distance 3D ne dépasse pas la portée radio. Le coût d'un lien augmente avec la distance.
- Le plan de contrôle détecte un changement immédiatement mais met 3 s à installer une nouvelle route. Les paquets qui utilisent entre-temps une route cassée sont perdus.

Ce délai volontaire rend observable la différence entre **détection**, **convergence** et **rétablissement du trafic**.

In [ ]:
def position(node: str, t: int) -> tuple[float, float, float]:
    """Position (x, y, altitude) en mètres à l'instant t."""
    if node == "G":
        return (0.0, 0.0, 0.0)
    if node == "S":
        return (300.0 + 4.0 * math.sin(0.11 * t), 5.0 * math.sin(0.15 * t), 80.0)
    if node == "R1":
        y = 0.0 if t < 20 or t >= 40 else 240.0
        return (150.0, y, 45.0)
    if node == "U1":
        return (220.0 + 2.0 * math.sin(0.09 * t), 90.0 + 3.0 * math.sin(0.18 * t), 75.0)
    if node == "U2":
        return (110.0, 90.0 + 2.0 * math.sin(0.12 * t), 45.0)
    raise KeyError(node)


def topology_at(t: int):
    positions = {node: position(node, t) for node in NODES}
    graph = {node: {} for node in NODES}
    for index, left in enumerate(NODES):
        for right in NODES[index + 1:]:
            distance_m = math.dist(positions[left], positions[right])
            if distance_m <= RADIO_RANGE_M:
                cost = 1.0 + (distance_m / RADIO_RANGE_M) ** 2
                graph[left][right] = graph[right][left] = cost
    return positions, graph


def shortest_path(graph, source: str, destination: str):
    """Dijkstra compact ; le tri rend les égalités reproductibles."""
    queue = [(0.0, source, (source,))]
    best_cost = {}
    while queue:
        cost, node, path = heapq.heappop(queue)
        if cost >= best_cost.get(node, float("inf")):
            continue
        best_cost[node] = cost
        if node == destination:
            return list(path), cost
        for neighbor, edge_cost in sorted(graph[node].items()):
            heapq.heappush(queue, (cost + edge_cost, neighbor, path + (neighbor,)))
    return None, float("inf")


def edge_set(graph):
    return frozenset(
        tuple(sorted((left, right)))
        for left, neighbors in graph.items()
        for right in neighbors
        if left < right
    )


def path_is_valid(path, graph) -> bool:
    return bool(path) and all(right in graph[left] for left, right in zip(path, path[1:]))


def percentile(values, fraction: float) -> float:
    ordered = sorted(values)
    index = (len(ordered) - 1) * fraction
    low, high = math.floor(index), math.ceil(index)
    if low == high:
        return ordered[low]
    return ordered[low] + (ordered[high] - ordered[low]) * (index - low)

In [ ]:
def run_simulation(seed: int = SEED):
    rng = random.Random(seed)
    rows, events, delays_ms = [], [], []
    delivered = lost = route_switches = 0

    _, graph = topology_at(0)
    route, _ = shortest_path(graph, SOURCE, DESTINATION)
    previous_edges = edge_set(graph)
    pending_installation = None
    events.append({"t": 0, "event": "initialisation", "route": route})

    for t in range(DURATION_S + 1):
        positions, graph = topology_at(t)
        current_edges = edge_set(graph)

        if t and current_edges != previous_edges:
            pending_installation = t + CONTROL_DELAY_S
            events.append({"t": t, "event": "topologie modifiée", "route": list(route)})

        if pending_installation is not None and t >= pending_installation:
            new_route, _ = shortest_path(graph, SOURCE, DESTINATION)
            if new_route != route:
                route = new_route
                route_switches += 1
                events.append({"t": t, "event": "route installée", "route": list(route)})
            pending_installation = None

        valid = path_is_valid(route, graph)
        delivered_this_second = 0
        hops = len(route) - 1 if valid else None

        if valid:
            route_distance_m = sum(
                math.dist(positions[left], positions[right])
                for left, right in zip(route, route[1:])
            )
            for _ in range(PACKETS_PER_S):
                # Modèle pédagogique : traitement par saut + longueur du trajet + léger jitter seedé.
                delay_ms = max(0.1, 3.0 + 1.8 * hops + 0.025 * route_distance_m + rng.uniform(-0.6, 0.6))
                delays_ms.append(delay_ms)
                delivered += 1
                delivered_this_second += 1
        else:
            lost += PACKETS_PER_S

        rows.append({
            "t": t,
            "route": tuple(route) if route else None,
            "valid": valid,
            "delivered": delivered_this_second,
            "lost": PACKETS_PER_S - delivered_this_second,
            "hops": hops,
        })
        previous_edges = current_edges

    sent = (DURATION_S + 1) * PACKETS_PER_S
    valid_hops = [row["hops"] for row in rows if row["hops"] is not None]
    metrics = {
        "sent": sent,
        "delivered": delivered,
        "lost": lost,
        "pdr_pct": 100.0 * delivered / sent,
        "mean_delay_ms": statistics.fmean(delays_ms),
        "p50_delay_ms": percentile(delays_ms, 0.50),
        "p95_delay_ms": percentile(delays_ms, 0.95),
        "route_switches": route_switches,
        "outage_s": sum(not row["valid"] for row in rows),
        "mean_hops": statistics.fmean(valid_hops),
        "max_hops": max(valid_hops),
    }
    signature_payload = {"metrics": metrics, "events": events}
    signature = hashlib.sha256(
        json.dumps(signature_payload, sort_keys=True, ensure_ascii=True).encode("utf-8")
    ).hexdigest()[:16]
    return {"metrics": metrics, "rows": rows, "events": events, "signature": signature}


result = run_simulation()

In [ ]:
m = result["metrics"]
print("=== Journal du plan de contrôle ===")
for event in result["events"]:
    route_text = " → ".join(event["route"]) if event["route"] else "aucune"
    print(f"t={event['t']:>2} s | {event['event']:<19} | {route_text}")

print("\n=== KPIs de bout en bout ===")
print(f"Paquets              : {m['delivered']} livrés / {m['sent']} envoyés ({m['lost']} perdus)")
print(f"PDR                  : {m['pdr_pct']:.2f} %")
print(f"Délai moyen / p50/p95: {m['mean_delay_ms']:.2f} / {m['p50_delay_ms']:.2f} / {m['p95_delay_ms']:.2f} ms")
print(f"Rupture observable   : {m['outage_s']} s")
print(f"Changements de route : {m['route_switches']}")
print(f"Sauts moyens / max   : {m['mean_hops']:.2f} / {m['max_hops']}")
print(f"Signature déterministe: {result['signature']}")

## 2. Des tests exécutables, pas seulement une jolie courbe

Les assertions suivantes transforment le scénario en petite expérience reproductible : elles vérifient la rupture, la perte transitoire, l'installation du chemin de secours, le retour au chemin court et l'identité d'une seconde exécution avec la même seed.

In [ ]:
rows = result["rows"]
assert rows[19]["route"] == ("S", "R1", "G") and rows[19]["valid"]
assert all(not rows[t]["valid"] for t in (20, 21, 22))
assert rows[23]["route"] == ("S", "U1", "U2", "G") and rows[23]["valid"]
assert rows[43]["route"] == ("S", "R1", "G") and rows[43]["valid"]
assert result["metrics"]["route_switches"] == 2
assert run_simulation(SEED)["signature"] == result["signature"]
print("✓ Rupture, convergence, reroutage, retour et reproductibilité vérifiés.")

In [ ]:
class InlineHTML:
    """Petit objet affichable par Jupyter, sans IPython ni bibliothèque graphique."""

    def __init__(self, markup: str):
        self.markup = markup

    def _repr_html_(self) -> str:
        return self.markup


def map_point(point):
    x, y, _ = point
    return 36.0 + (x + 25.0) / 355.0 * 348.0, 258.0 - (y + 30.0) / 300.0 * 220.0


def topology_svg(t: int, row) -> str:
    positions, graph = topology_at(t)
    route = row["route"] or ()
    route_edges = {frozenset((left, right)) for left, right in zip(route, route[1:])}
    parts = [
        '<svg viewBox="0 0 420 300" role="img" aria-label="Topologie FANET">',
        '<rect width="420" height="300" rx="18" fill="#F8FAFC"/>',
    ]
    for value in range(40, 381, 55):
        parts.append(f'<line x1="{value}" y1="42" x2="{value}" y2="258" stroke="#E2E8F0"/>')
    for value in range(42, 259, 43):
        parts.append(f'<line x1="36" y1="{value}" x2="384" y2="{value}" stroke="#E2E8F0"/>')

    for left, right in sorted(edge_set(graph)):
        x1, y1 = map_point(positions[left])
        x2, y2 = map_point(positions[right])
        active = frozenset((left, right)) in route_edges
        color, width = ("#16A34A", 5) if active else ("#CBD5E1", 2)
        parts.append(f'<line x1="{x1:.1f}" y1="{y1:.1f}" x2="{x2:.1f}" y2="{y2:.1f}" stroke="{color}" stroke-width="{width}" stroke-linecap="round"/>')

    for left, right in zip(route, route[1:]):
        if right not in graph[left]:
            x1, y1 = map_point(positions[left])
            x2, y2 = map_point(positions[right])
            parts.append(f'<line x1="{x1:.1f}" y1="{y1:.1f}" x2="{x2:.1f}" y2="{y2:.1f}" stroke="#DC2626" stroke-width="5" stroke-dasharray="9 7"/>')

    for node, point in positions.items():
        x, y = map_point(point)
        altitude = point[2]
        color = "#0F172A" if node == "G" else ("#2563EB" if node == "S" else "#F59E0B")
        if node == "G":
            parts.append(f'<rect x="{x - 9:.1f}" y="{y - 9:.1f}" width="18" height="18" rx="4" fill="{color}" stroke="white" stroke-width="3"/>')
        else:
            parts.append(f'<circle cx="{x:.1f}" cy="{y:.1f}" r="10" fill="{color}" stroke="white" stroke-width="3"/>')
        parts.append(f'<text x="{x + 13:.1f}" y="{y - 10:.1f}" font-size="12" font-weight="700" fill="#0F172A">{escape(node)}</text>')
        parts.append(f'<text x="{x + 13:.1f}" y="{y + 5:.1f}" font-size="10" fill="#64748B">z={altitude:.0f} m</text>')

    status = "ROUTE VALIDE" if row["valid"] else "ROUTE CASSÉE"
    badge = "#DCFCE7" if row["valid"] else "#FEE2E2"
    ink = "#166534" if row["valid"] else "#991B1B"
    route_text = escape(" → ".join(route) if route else "aucune route")
    parts.append(f'<rect x="18" y="12" width="116" height="24" rx="12" fill="{badge}"/>')
    parts.append(f'<text x="76" y="28" text-anchor="middle" font-size="10" font-weight="800" fill="{ink}">{status}</text>')
    parts.append(f'<text x="402" y="28" text-anchor="end" font-size="13" font-weight="800" fill="#0F172A">t={t} s</text>')
    parts.append(f'<text x="210" y="286" text-anchor="middle" font-size="11" fill="#334155">{route_text}</text>')
    parts.append('</svg>')
    return "".join(parts)


def timeline_svg(rows) -> str:
    x = lambda t: 58.0 + t / DURATION_S * 804.0
    y_delivery = lambda value: 188.0 - value / PACKETS_PER_S * 118.0
    y_hops = lambda value: 188.0 - value / 4.0 * 118.0

    delivery_points = [(x(rows[0]["t"]), y_delivery(rows[0]["delivered"]))]
    hop_points = [(x(rows[0]["t"]), y_hops(rows[0]["hops"] or 0))]
    for previous, current in zip(rows, rows[1:]):
        current_x = x(current["t"])
        delivery_points.extend(((current_x, y_delivery(previous["delivered"])), (current_x, y_delivery(current["delivered"]))))
        hop_points.extend(((current_x, y_hops(previous["hops"] or 0)), (current_x, y_hops(current["hops"] or 0))))

    delivery_path = " ".join(f"{px:.1f},{py:.1f}" for px, py in delivery_points)
    hop_path = " ".join(f"{px:.1f},{py:.1f}" for px, py in hop_points)
    parts = [
        '<svg viewBox="0 0 920 245" role="img" aria-label="Livraison et sauts au cours du temps">',
        '<rect width="920" height="245" rx="18" fill="#F8FAFC"/>',
        f'<rect x="{x(20):.1f}" y="42" width="{x(23) - x(20):.1f}" height="154" fill="#FEE2E2"/>',
    ]
    for tick in range(0, DURATION_S + 1, 10):
        tick_x = x(tick)
        parts.append(f'<line x1="{tick_x:.1f}" y1="42" x2="{tick_x:.1f}" y2="196" stroke="#E2E8F0"/>')
        parts.append(f'<text x="{tick_x:.1f}" y="214" text-anchor="middle" font-size="10" fill="#64748B">{tick} s</text>')
    for value in (0, 5, 10):
        tick_y = y_delivery(value)
        parts.append(f'<line x1="58" y1="{tick_y:.1f}" x2="862" y2="{tick_y:.1f}" stroke="#E2E8F0"/>')
        parts.append(f'<text x="46" y="{tick_y + 4:.1f}" text-anchor="end" font-size="10" fill="#64748B">{value}</text>')

    parts.append(f'<polyline points="{delivery_path}" fill="none" stroke="#16A34A" stroke-width="5" stroke-linejoin="round"/>')
    parts.append(f'<polyline points="{hop_path}" fill="none" stroke="#7C3AED" stroke-width="3" stroke-dasharray="8 5"/>')
    for t, label, color in ((20, "rupture", "#DC2626"), (23, "secours", "#D97706"), (43, "retour", "#2563EB")):
        event_x = x(t)
        parts.append(f'<circle cx="{event_x:.1f}" cy="28" r="5" fill="{color}"/>')
        parts.append(f'<text x="{event_x:.1f}" y="17" text-anchor="middle" font-size="10" font-weight="700" fill="{color}">{label}</text>')
    parts.extend((
        '<line x1="620" y1="228" x2="648" y2="228" stroke="#16A34A" stroke-width="4"/><text x="654" y="232" font-size="10" fill="#334155">paquets livrés/s</text>',
        '<line x1="754" y1="228" x2="782" y2="228" stroke="#7C3AED" stroke-width="3" stroke-dasharray="6 4"/><text x="788" y="232" font-size="10" fill="#334155">sauts</text>',
        '</svg>',
    ))
    return "".join(parts)


def build_dashboard(result) -> str:
    metrics = result["metrics"]
    event_chips = "".join(
        f'<span class="fanet-event"><b>t={event["t"]} s</b>{escape(event["event"])}<small>{escape(" → ".join(event["route"]))}</small></span>'
        for event in result["events"]
    )
    cards = (("PDR", f"{metrics['pdr_pct']:.2f} %", "livraison globale"),
             ("DÉLAI P95", f"{metrics['p95_delay_ms']:.2f} ms", "paquets reçus"),
             ("INDISPONIBILITÉ", f"{metrics['outage_s']} s", "convergence"),
             ("REROUTAGES", str(metrics["route_switches"]), "secours puis retour"))
    card_html = "".join(
        f'<div class="fanet-kpi"><span>{label}</span><strong>{value}</strong><small>{note}</small></div>'
        for label, value, note in cards
    )
    style = """
    <style>
      .fanet-dashboard{font-family:Inter,ui-sans-serif,system-ui,-apple-system,sans-serif;color:#0f172a;background:linear-gradient(145deg,#eef6ff,#f8fafc 48%,#ecfdf5);border:1px solid #dbeafe;border-radius:24px;padding:22px;box-shadow:0 16px 45px rgba(15,23,42,.10)}
      .fanet-head{display:flex;justify-content:space-between;gap:16px;align-items:flex-start;margin-bottom:16px}.fanet-head h2{font-size:23px;margin:0 0 5px}.fanet-head p{color:#475569;margin:0}.fanet-seed{font:700 11px ui-monospace,monospace;background:#0f172a;color:#e2e8f0;border-radius:999px;padding:8px 12px;white-space:nowrap}
      .fanet-kpis{display:grid;grid-template-columns:repeat(4,minmax(0,1fr));gap:10px;margin:14px 0}.fanet-kpi{background:rgba(255,255,255,.92);border:1px solid #e2e8f0;border-radius:16px;padding:13px}.fanet-kpi span,.fanet-kpi small{display:block;color:#64748b;font-size:10px}.fanet-kpi strong{display:block;font-size:24px;margin:4px 0;color:#0f172a}
      .fanet-topologies{display:grid;grid-template-columns:1fr 1fr;gap:12px}.fanet-panel{background:white;border:1px solid #e2e8f0;border-radius:19px;padding:8px}.fanet-panel h3{font-size:13px;margin:4px 8px;color:#334155}.fanet-panel svg{width:100%;height:auto;display:block}.fanet-chart{margin-top:12px}
      .fanet-events{display:flex;gap:8px;overflow-x:auto;margin-top:12px;padding-bottom:2px}.fanet-event{min-width:132px;background:#fff;border-left:4px solid #3b82f6;border-radius:10px;padding:8px 10px;font-size:11px;color:#334155}.fanet-event b,.fanet-event small{display:block}.fanet-event small{color:#64748b;margin-top:3px}
      .fanet-foot{display:flex;justify-content:space-between;gap:12px;margin-top:13px;color:#475569;font-size:11px}.fanet-foot code{color:#1d4ed8}
      @media(max-width:760px){.fanet-kpis{grid-template-columns:1fr 1fr}.fanet-topologies{grid-template-columns:1fr}.fanet-head{display:block}.fanet-seed{display:inline-block;margin-top:10px}}
    </style>
    """
    return style + f"""
    <section class="fanet-dashboard">
      <header class="fanet-head"><div><h2>Mini-FANET · rupture → convergence → secours</h2><p>Le plan de données révèle les trois secondes pendant lesquelles le plan de contrôle converge.</p></div><span class="fanet-seed">SEED {SEED}</span></header>
      <div class="fanet-kpis">{card_html}</div>
      <div class="fanet-topologies">
        <div class="fanet-panel"><h3>Avant la rupture · chemin court</h3>{topology_svg(19, result['rows'][19])}</div>
        <div class="fanet-panel"><h3>Après convergence · corridor de secours</h3>{topology_svg(25, result['rows'][25])}</div>
      </div>
      <div class="fanet-panel fanet-chart"><h3>Livraison et longueur de route · 0 à {DURATION_S} s</h3>{timeline_svg(result['rows'])}</div>
      <div class="fanet-events">{event_chips}</div>
      <footer class="fanet-foot"><span>Vert : route active · gris : liens disponibles · violet : nombre de sauts</span><code>sha256:{result['signature']}</code></footer>
    </section>
    """


dashboard_markup = build_dashboard(result)
assert dashboard_markup.count("<svg") == 3
assert result["signature"] in dashboard_markup
fanet_dashboard = InlineHTML(dashboard_markup)
fanet_dashboard

## 3. Ce que l'expérience permet de comprendre

À 20 s, `R1` sort de portée : la topologie réelle a déjà changé, mais la route installée pointe encore vers ce relais. Les trois secondes sans livraison matérialisent le **temps de convergence**. À 23 s, le corridor `S → U1 → U2 → G` rétablit le trafic au prix d'un saut supplémentaire et d'un délai légèrement supérieur. À 43 s, le chemin court est réinstallé.

Le **PDR** agrège ici le coût visible de cette transition. Le **p95 de délai** décrit un quantile des paquets effectivement livrés ; il ne doit pas masquer les paquets perdus pendant la rupture. Il faut donc lire les KPIs ensemble.

### Limites et passage à l'échelle

Ce notebook ne modélise ni interférences, ni MAC 802.11, ni énergie, ni antennes, ni dynamique de vol. Son algorithme SPF générique ne prétend pas être AODV, OLSR, BATMAN ou GPSR. Pour une étude scientifique, on conserverait cette seed et ces assertions comme oracle pédagogique, puis on comparerait des protocoles réels dans **ns-3** ou **OMNeT++/INET** ; pour l'émulation, on pourrait exécuter de vraies piles dans **CORE/EMANE** ou **Mininet-WiFi**, et coupler la mobilité à **PX4/ArduPilot + Gazebo/ROS 2**.

Questions utiles pour prolonger le TP : quel protocole minimise l'indisponibilité ? Quel compromis entre fréquence des mises à jour, énergie, PDR et stabilité de route ? Que devient le résultat avec plusieurs flux, des pertes radio ou une panne simultanée de deux relais ?